In [ ]:
!pip install sentencepiece tiktoken evaluate
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 84.8 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1


In [ ]:
import pandas as pd
from google.colab import drive
from transformers import AutoTokenizer, BartForConditionalGeneration, BartTokenizer, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import Dataset
import torch
import evaluate

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
splits = ['Train', 'Val', 'Test']
dfs = {}

for split in splits:
  path = f'/content/drive/MyDrive/266_Final/data/{split}/wikidt_{split.lower()}_tabular.parquet'
  dfs[split] = pd.read_parquet(path)
  print(f'{split} shape: {dfs[split].shape}')

train_df, val_df, test_df = dfs['Train'], dfs['Val'], dfs['Test']

Train shape: (53492, 10)
Val shape: (6686, 10)
Test shape: (6687, 10)


In [ ]:
train_df.head()

,question,answer,context,pairs,positive_sentences,image,context_tables_json,num_tables,context_multitabqa,context_tapex
0,geno hayes and nick roach both played which po...,no answer,Position Player $ Free agency tag Date signed ...,"[{'label': 0, 'query': 'geno hayes and nick ro...",0,204_783_page.png,"{""tables"":[{""table_name"":""free_agents"",""header...",3,free_agents : col : position | player | $ | fr...,free_agents : col : position | player | $ | fr...
1,What is the number of bronze for Canada and si...,no answer,"Pos Team Time Gold Switzerland (Reto Götschi, ...","[{'label': 0, 'query': 'What is the number of ...",0,11338408_page.png,"{""tables"":[{""table_name"":""two_man_bobsleigh"",""...",4,two_man_bobsleigh : col : pos | team | time | ...,two_man_bobsleigh : col : pos | team | time | ...
2,"What is Date, when Opponent is ""Philadelphia 7...",february 16,# Date H/A/N Opponent W/L Score Record 40 Janu...,"[{'label': 1, 'query': 'What is Date, when Opp...",1,16946097_page.png,"{""tables"":[{""table_name"":""game_results"",""heade...",1,game_results : col : date | h/a/n | opponent |...,col : date | h/a/n | opponent | w/l | score | ...
3,Which Class has a Weight of 203?,sr,Position Number Name Height Weight Class Homet...,"[{'label': 1, 'query': 'Which Class has a Weig...",29,14624447_page.png,"{""tables"":[{""table_name"":""defensive_players"",""...",6,defensive_players : col : position | number | ...,defensive_players : col : position | number | ...
4,Who had 0 total votes in the purple team?,eliminated week 12 - the walk,Members Teammate Couples Team Blue Red 1 Blue ...,"[{'label': 1, 'query': 'Who had 0 total votes ...",1,18426579_page.png,"{""tables"":[{""table_name"":""castaway_elimination...",1,castaway_eliminations : col : members | teamma...,col : members | teammate | couples | team | bl...


In [ ]:
train_df['context_tapex'][45000]

'seat_info : col : seat | state | majority | member | party row 1 : hinkler | qld | 00.0 | paul neville | npa row 2 : solomon | nt | 00.1 | dave tollner | lib row 3 : adelaide | sa | 00.2 | hon trish worth | lib row 4 : canning | wa | 00.4 | don randall | lib row 5 : dobell | nsw | 00.4 | ken ticehurst | lib row 6 : parramatta | nsw | 01.1 | ross cameron | lib row 7 : mcewen | vic | 01.2 | fran bailey | lib row 8 : paterson | nsw | 01.4 | bob baldwin | lib row 9 : herbert | qld | 01.6 | peter lindsay | lib row 10 : richmond | nsw | 01.6 | hon larry anthony | npa row 11 : deakin | vic | 01.7 | philip barresi | lib row 12 : eden-monaro | nsw | 01.7 | gary nairn | lib row 13 : hindmarsh | sa | 01.9 | hon christine gallus | lib election_results : col : candidate | votes | percentage row 1 : white (democrat) | 6,000 | 06.0% row 2 : smith (labor) | 45,000 | 45.0% row 3 : jones (liberal) | 35,000 | 35.0% row 4 : johnson (green) | 10,000 | 10.0% row 5 : davies (ind) | 4,000 | 04.0%'

In [ ]:
model = "microsoft/tapex-base"

In [ ]:
tokenizer = BartTokenizer.from_pretrained(model, use_fast=False)
model = BartForConditionalGeneration.from_pretrained(model)

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/988 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def preprocess_function(examples):
  inputs = [
        f"{q.strip()} {c.strip()}"
        for q, c in zip(examples["question"], examples["context_tapex"])
  ]

  model_inputs = tokenizer(
        inputs,
        max_length=1024,
        padding=False,
        truncation=True,
    )

  labels = tokenizer(
        text_target=examples["answer"],
        max_length=1024,
        padding=False,
        truncation=True,
    )


  pad_id = tokenizer.pad_token_id
  model_inputs["labels"] = [
      [(token if token != pad_id else -100) for token in seq]
      for seq in labels["input_ids"]
  ]

  model_inputs["labels"] = labels["input_ids"]
  return model_inputs




In [ ]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
tokenized_train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_val_dataset = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names,
)

tokenized_test_dataset = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names,
)

Map:   0%|          | 0/53492 [00:00<?, ? examples/s]

Map:   0%|          | 0/6686 [00:00<?, ? examples/s]

Map:   0%|          | 0/6687 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

In [ ]:
tokenized_train_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 53492
})

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./tapex-base-paper",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=32,
    eval_strategy="epoch",
    learning_rate=4e-5,
    num_train_epochs=10,
    predict_with_generate=True,
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    report_to="none",
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    data_collator=data_collator,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.497801
2,No log,0.450061
3,18.503002,0.449640
4,18.503002,0.436695
5,10.386318,0.448736
6,10.386318,0.444964
7,10.386318,0.452843
8,7.498234,0.460263
9,7.498234,0.470981
10,5.864493,0.471093


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2090, training_loss=10.339607523502915, metrics={'train_runtime': 2786.9058, 'train_samples_per_second': 191.94, 'train_steps_per_second': 0.75, 'total_flos': 3.254574978282701e+17, 'train_loss': 10.339607523502915, 'epoch': 10.0})

In [ ]:
test_output = trainer.predict(
    test_dataset=tokenized_test_dataset
)

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1977: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


In [ ]:
for k, v in test_output.metrics.items():
    print(f"{k}: {v}")

test_loss: 0.48374617099761963
test_runtime: 73.9407
test_samples_per_second: 90.437
test_steps_per_second: 11.306


In [ ]:
raw_preds = test_output.predictions

In [ ]:
predicted_answers = tokenizer.batch_decode(raw_preds, skip_special_tokens=True)

In [ ]:
test_results_df = pd.DataFrame({
    "question": test_df["question"].values,
    "context": test_df["context"].values,
    "ground_truth": test_df["answer"].astype(str).values,
    "predicted": [p.strip() for p in predicted_answers],
})

In [ ]:
test_results_df["is_exact_match"] = (
    test_results_df["predicted"].str.strip().str.lower()
    == test_results_df["ground_truth"].str.strip().str.lower()
)

In [ ]:
test_results_df[test_results_df["is_exact_match"] == True]

,question,context,ground_truth,predicted,is_exact_match
0,What are the title(s) of episodes directed by ...,No. overall No. in season Title Directed by Wr...,buffy vs. dracula,buffy vs. dracula,True
1,who was the top driver at the 1967 british gra...,Pos No Driver Constructor Laps Time/Retired Gr...,no answer,no answer,True
2,"What is Years, when Postition is PG, and when ...",Player Nationality Jersey Number(s) Position Y...,1977-1984,1977-1984,True
3,How many attendance numbers had more than 15 g...,Season Record Pct. Games Attendance Average Se...,1,1,True
4,which rider listed is the only one not to ride...,Year Rider Victories Bike Manufacturer's Champ...,no answer,no answer,True
...,...,...,...,...,...
6682,What tournament had a Runner(s)-up of Tom Kite?,No. Year Tournament Opponent(s) Result 1 1987 ...,charles schwab cup championship,charles schwab cup championship,True
6683,What's the total percent of the $40-50K ones t...,Data ³³ Headcount Percentage of total Less tha...,46.5%,46.5%,True
6684,Name the points classification for levi leiphe...,Cyclist Team Time 1 Fabian Cancellara (SUI) Le...,peter sagan,peter sagan,True
6685,what is the score when the surface is clay and...,Legend Grand Slam 0 WTA Championships 0 Tier I...,"7-6, 7-5","7-6, 7-5",True


In [ ]:
test_results_df[test_results_df["is_exact_match"] == False]

,question,context,ground_truth,predicted,is_exact_match
5,What is the latest year when Phil Anderson won?,Year Race name Rider status Winner Team/Countr...,1993,2008,False
7,"How many Played has a % Win of 50.00%, and Los...",Year Played Wins Losses No Result % Win T20 Cu...,0,no answer,False
10,What is the number for combaya municipality wh...,Language Sorata Municipality Guanay Municipali...,20.0,20,False
13,Which Away team played when they had a Crowd o...,Home team Home team score Away team Away team ...,richmond,carlton,False
15,"In the games at corio oval, what was the highe...",Home team Home team score Away team Away team ...,10500,15000,False
...,...,...,...,...,...
6647,Where was the player born when the appearances...,"1. A Triggs, Roger (1984). Gillingham Football...",england,newcomer,False
6659,"What is Record, when Game is ""36""?",Date High rebounds High assists Location Atten...,28-8,8 - 7,False
6660,What is the Place of the Player from New Zealand?,Player Country Year(s) won R1 R2 Total To par ...,t3,t5,False
6678,Who is the GamePlan when Arabella Lazenby is t...,Actor in original production Actor required Ga...,lynette saxon,micky rale,False


In [ ]:
test_results_df[test_results_df["is_exact_match"] == False]['context'][6678]

'Actor in original production Actor required GamePlan FlatSpin RolePlay Bill Champion Male, younger Troy Stephens Sam Berryman Justin Lazenby Saskia Butler Female, younger Sorrel Saxon Tracy Taylor Julie-Ann Jobson Alison Pargeter Female, younger Kelly Butcher Rosie Seymour Paige Petite Tim Faraday Male, older Dan Endicott Tommy Angel Micky Rale Robert Austin Male, older Leo Tyler Maurice Whickett Derek Jobson Beth Tuckley Female, older Grace Page Edna Stricken Dee Jobson Jacqueline King Female, older Lynette Saxon Annette Sefton- Wilcox Arabella Lazenby'

In [ ]:
em_metric = evaluate.load("exact_match")

results = em_metric.compute(
    predictions=test_df["answer"].astype(str).values,
    references=[p.strip() for p in predicted_answers],
    ignore_case=True,
    ignore_punctuation=True,
)

In [ ]:
print(f"EM: {results['exact_match'] * 100:.2f}%")

EM: 75.55%
